In [ ]:
import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from typing import List, Dict, Tuple
from scipy import stats
from sklearn.metrics import roc_curve, auc
import json
# Add necessary paths
sys.path.append('.')
sys.path.append('./plotting_notebooks')
MAIN_AGENT = 'claude-3-7-sonnet-20250219'
# MAIN_AGENT = 'claude-3-5-sonnet-20241022'
MONITOR_AGENT = 'claude-3-5-sonnet-20241022'
TRUSTED_AGENT = 'claude-3-5-haiku-20241022'
# TRUSTED_AGENT = 'gpt-4o'
NUM_TASKS_THRESHOLD = 8
EXCLUDE_TASKS = []
# Import the loading functions from the analysis script
from protocol_plotting_utils import (
    load_in_combined_results,
    load_protocol_intervention_transcripts,
    load_protocol_intervention_data,
    prepare_protocol_monitoring_data,
    merge_protocol_monitoring_with_combined_results,
    make_protocol_results_and_data,
    get_common_tasks,
    filter_protocol_results_and_data,
    prepare_both_system_prompt_data,
    get_uninterrupted_usefulness,
    make_useful_data,
    calculate_task_mean_with_se,
    wilson_conf_interval,
    get_p_doom
)
from plotting_notebooks.plotting_utils import (
    prepare_monitoring_data
)
from simulate_markov_utils import plot_markov_df, create_markov_df, add_jitter_to_column
from utils.analysis_utils import balance_classes_with_oversampling, extract_verdict

def parse_monitor_evaluation_files(base_dir: str) -> List[Dict]:
    """
    Parses JSON evaluation files recursively from a base directory.

    Args:
        base_dir: The root directory to search for JSON files, 
                  e.g., 'monitor_responses/rolled_out_evaluations'.

    Returns:
        A list of dictionaries, where each dictionary contains parsed 
        information for one JSON file.
    """
    results = []
    base_dir = os.path.abspath(base_dir) # Ensure absolute path for relpath
    print(f"Starting search in absolute base directory: {base_dir}") # Debug print
    for root, _, files in os.walk(base_dir):
        for filename in files:
            if filename.endswith('.json'):
                file_path = os.path.join(root, filename)
                try:
                    # Calculate relative path and split into components
                    relative_path = os.path.relpath(file_path, base_dir)
                    path_parts = relative_path.split(os.sep)
                    
                    # Extract info from path structure
                    # Example: task/transcript_type/protocol/main_agent/monitor_agent/file.json
                    if len(path_parts) >= 6: # Ensure sufficient path depth
                        task_name = path_parts[0]
                        transcript_type = path_parts[1]
                        protocol_name = path_parts[2]
                        main_agent = path_parts[3]
                        monitor_agent = path_parts[4]
                        file_name = path_parts[5]
                        # main_agent = path_parts[3] # Could extract if needed
                        # monitor_agent_in_path = path_parts[4] # Could extract if needed
                    else:
                        # Handle unexpected path structure, maybe log or skip
                        print(f"Skipping unexpected path structure: {file_path}")
                        continue

                    # Read and parse JSON content
                    with open(file_path, 'r') as f:
                        data = json.load(f)

                    # Extract info from JSON data
                    monitor_model = data.get('monitor_model')
                    monitor_response = data.get('monitor_response')
                    verdict = data.get('verdict')
                    usage = data.get('usage')

                    # Store extracted data
                    record = {
                        'task_name': task_name,
                        'transcript_type': transcript_type,
                        'protocol_name': protocol_name,
                        'main_agent': main_agent,
                        'monitor_agent': monitor_agent,
                        'file_name': file_name,
                        'monitor_model': monitor_model,
                        'monitor_response': monitor_response,
                        'rolled_out_suspicion_score': verdict,
                        'usage': usage,
                        'file_path': file_path # Include file path for reference
                    }
                    results.append(record)

                except json.JSONDecodeError:
                    print(f"Error decoding JSON from file: {file_path}")
                except KeyError as e:
                    print(f"Missing key {e} in file: {file_path}")
                except Exception as e:
                    print(f"An unexpected error occurred processing file {file_path}: {e}")
                    
    return results

monitor_evaluation_files = parse_monitor_evaluation_files('../monitor_responses/rolled_out_evaluations')
monitor_evaluation_files = pd.DataFrame(monitor_evaluation_files)

In [ ]:
protocol_colors = {
    'claude_take_the_wheel_no_notes': '#a6cee3',  # light blue
    'defer_to_trusted': '#377eb8',  # blue
    'resample_sequential': '#c7c7c7',  # gray
    'resample_then_defer': '#17becf',  # cyan
    'resample_sequential_rolled_out': '#ff7f00',  # orange
    'defer_to_trusted_rolled_out': '#9467bd',  # purple
}
control_protocol_list = protocol_colors.keys()

protocol_results_and_data = make_protocol_results_and_data(control_protocol_list, MAIN_AGENT, TRUSTED_AGENT, base_dir='protocol_results')

In [ ]:
merged_df = pd.merge(protocol_results_and_data, monitor_evaluation_files, on=['task_name', 'transcript_type', 'protocol_name', 'main_agent', 'monitor_agent', 'file_name'])

In [ ]:
print(merged_df.main_agent.unique())
print(merged_df.monitor_agent.unique())
for protocol in control_protocol_list:
    filtered_merged = merged_df[merged_df.protocol_name == protocol]
    plt.figure(figsize=(10, 6))
    for metric in ['back_0.2_max_suspicion_score', 'suspiciousness', 'rolled_out_suspicion_score']:
        fpr, tpr, thresholds = roc_curve(filtered_merged.transcript_type != 'transcripts_no_secondary_task', filtered_merged[metric])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{metric} (AUC = {roc_auc:.2f})')
    plt.legend()
    plt.title(f'ROC Curve for {protocol}')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.grid(True, alpha=0.5)
    plt.plot([0, 1], [0, 1], 'k--', label='Random')
    plt.show()